In [1]:
import os
import openai # Importamos openai directo para capturar sus errores específicos
from openai import OpenAI
from dotenv import load_dotenv

# 1. Modularización: Todo el comportamiento del Asistente se encapsula en una Clase
class AsistenteMetro:
    def __init__(self):
        load_dotenv()
        self.client = OpenAI(
            base_url=os.environ.get("GITHUB_BASE_URL"),
            api_key=os.environ.get("GITHUB_TOKEN")
        )
        # Límite: Solo recordará las últimas 3 preguntas y 3 respuestas (6 en total)
        self.limite_mensajes = 6 
        self.memoria = []
        self._inicializar_memoria()

    def _inicializar_memoria(self):
        """Prepara el RAG simulado y el prompt base del sistema"""
        contexto_red = """
        INFORMACIÓN DE TARIFAS (General):
        - Horario Punta (07:00-08:59 y 18:00-19:59): $840
        - Horario Valle (09:00-17:59 y 20:00-20:44): $760
        - Horario Bajo (06:00-06:59 y 20:45-23:00): $680
        - Tarifa Estudiante (TNE) y Adulto Mayor: $240 en todo horario.

        ESTRUCTURA DE LÍNEAS Y COMBINACIONES CLAVE:
        - Línea 1 (Roja): Terminales San Pablo / Los Dominicos. Combina con L2, L3, L4 (Tobalaba), L5 y L6.
        - Línea 4 (Azul): Terminales Tobalaba / Plaza Puente Alto. Combina con L1 (Tobalaba), L3 y L5.
        """
        horario_actual = "Día hábil, 18:30 hrs (Horario Punta)"
        alerta_operativa = "Línea 1 presenta retrasos por asistencia a pasajero en estación Baquedano."

        prompt_sistema = f"""
        Actúa como el asistente virtual oficial de servicio al cliente de Metro de Santiago.
        Contexto de la red: {contexto_red}
        Horario actual: {horario_actual}
        Alertas operativas: {alerta_operativa}

        Reglas:
        1. Indica la línea inicial, combinaciones y estación final.
        2. Informa la tarifa exacta del viaje.
        3. Advierte sobre alertas operativas.
        """
        # El primer mensaje (índice 0) siempre es el contexto intocable del sistema
        self.memoria = [{"role": "system", "content": prompt_sistema}]

    def _gestionar_tokens(self):
        """2. Control de Memoria: Evita que el historial crezca infinitamente"""
        # Si la memoria supera el límite, conservamos el prompt_sistema [0] y cortamos el resto
        if len(self.memoria) > self.limite_mensajes + 1:
            self.memoria = [self.memoria[0]] + self.memoria[-self.limite_mensajes:]

    def consultar_ruta(self, pregunta_usuario):
        """Método principal para recibir preguntas y retornar respuestas"""
        self.memoria.append({"role": "user", "content": pregunta_usuario})
        self._gestionar_tokens() # Limpiamos la memoria antes de gastar tokens

        try:
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=self.memoria,
                temperature=0.7,
                max_tokens=600
            )
            respuesta = response.choices[0].message.content
            self.memoria.append({"role": "assistant", "content": respuesta})
            return respuesta

        # 3. Manejo de Errores Específicos
        except openai.RateLimitError:
            return "⚠️ El sistema está recibiendo muchas consultas. Por favor, intenta en unos segundos."
        except openai.APIConnectionError:
            return "🔌 Hay un problema conectando con el servidor. Revisa tu internet."
        except Exception as e:
            # Error genérico por si falla cualquier otra cosa
            return f"❌ Ocurrió un error inesperado en el sistema: {str(e)}"

# ==========================================
# Ejecución limpia e independiente
# ==========================================
if __name__ == "__main__":
    # Instanciamos nuestro objeto Asistente
    asistente = AsistenteMetro()
    print("🚇 Asistente de Metro iniciado (Escribe 'salir' para terminar)\n")

    while True:
        texto_usuario = input("Tú: ")
        if texto_usuario.lower() == 'salir':
            print("Asistente: ¡Buen viaje! Cerrando sistema.")
            break
            
        # Toda la complejidad queda oculta detrás de esta única llamada
        respuesta_final = asistente.consultar_ruta(texto_usuario)
        print(f"\nAsistente: {respuesta_final}\n")

🚇 Asistente de Metro iniciado (Escribe 'salir' para terminar)


Asistente: ¡Hola! Soy el asistente virtual oficial de servicio al cliente de Metro de Santiago. ¿En qué puedo ayudarte hoy? 😊



KeyboardInterrupt: Interrupted by user